# PEEK ResNet50 Dense-Head Demo

This notebook launches the Imagenette2 fine-tuning job for the ResNet50 one-dense-head model, then runs the same compact trained-vs-random-head PEEK geometry study for the diagram set.


In [ ]:
from pathlib import Path
import csv
import numpy as np

from peek import (
    GeometryConfig,
    ImagenetteFineTuneConfig,
    compare_geometry_csvs,
    launch_imagenette_training_tmux,
    plot_dense_head_alignment_maps,
    render_geometry_directory,
    run_dense_head_analytic_study,
    train_imagenette_model,
)


Training weights are written to `weights/`. Analytic outputs are written under `analysis/resnet50_dense_head/` relative to the repo root.


In [ ]:
train_config = ImagenetteFineTuneConfig(
    model_name="resnet50",
    data_root="../datasets/imagenette2",
    checkpoint_out="weights/resnet50_one_dense_nopool_imagenette_best.pt",
    last_checkpoint_out="weights/resnet50_one_dense_nopool_imagenette_last.pt",
    history_out="analysis/resnet50_dense_head/train_history.csv",
    batch_size=64,
    epochs=50,
    head_warmup_epochs=1,
    learning_rate=3e-4,
    weight_decay=1e-4,
    label_smoothing=0.10,
    dropout=0.2,
    early_stop_patience=8,
    grad_clip_norm=1.0,
    random_seed=1337,
    num_workers=4,
    amp=True,
    freeze_backbone_during_warmup=True,
)


Launch or re-run training in a detached `tmux` session if the weights are not already present.


In [ ]:
tmux_job = launch_imagenette_training_tmux(
    train_config,
    session_name="peek_resnet50_dense_head_imagenette",
)

tmux_job


If you want to train in the foreground instead, use the next cell.


In [ ]:
# train_outputs = train_imagenette_model(train_config)
# train_outputs


Run the analytic study twice: once for a one-image-per-class diagram subset, and once across the full validation set for the numerical summaries.


In [ ]:
config = GeometryConfig(
    p_keep=0.20,
    topk_min=16,
    topk_frac=0.10,
    eps_z_l=1e-3,
    eps_z_sig=1e-3,
    grid_steps=9,
    alpha_max=1e-3,
    beta_max=1e-3,
)

diagram_outputs = run_dense_head_analytic_study(
    model_name="resnet50",
    val_dir="../datasets/imagenette2/val",
    checkpoint_path=train_config.checkpoint_out,
    out_dir="analysis/resnet50_dense_head/diagram_subset",
    one_per_class=True,
    config=config,
    save_npz=True,
)

full_outputs = run_dense_head_analytic_study(
    model_name="resnet50",
    val_dir="../datasets/imagenette2/val",
    checkpoint_path=train_config.checkpoint_out,
    out_dir="analysis/resnet50_dense_head/full_val",
    one_per_class=False,
    config=config,
    save_npz=False,
)

comparison = compare_geometry_csvs(
    full_outputs["train_csv"],
    full_outputs["random_csv"],
)

comparison[:5]


Render the saved trained/random diagram set to PNGs and preview one trained example inline.


In [ ]:
rendered = render_geometry_directory(
    diagram_outputs["map_dir"],
    "analysis/resnet50_dense_head/rendered_maps",
)

with open(Path("analysis/resnet50_dense_head") / "comparison_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=comparison[0].keys())
    writer.writeheader()
    writer.writerows(comparison)

rendered[:4]


In [ ]:
example_npz = sorted(Path(diagram_outputs["map_dir"]).glob("*.npz"))[0]
with np.load(example_npz, allow_pickle=True) as data:
    example_image_path = str(data["image_path"].item())

plot_dense_head_alignment_maps(
    image_path=example_image_path,
    npz_path=example_npz,
    variant="trained",
)
